# Build Retrieval Texts

combine game metadata with selected review summaries to create one `retrieval_text` per game

In [1]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_colwidth", 120)

DB_PATH = Path("steam_games_reviews_25.sqlite")
REVIEW_SUMMARIES_PATH = Path("review_summaries.csv")
OUTPUT_PATH = Path("games_retrieval_texts.csv")

assert DB_PATH.exists(), f"Database not found: {DB_PATH.resolve()}"
assert REVIEW_SUMMARIES_PATH.exists(), f"Review summaries not found: {REVIEW_SUMMARIES_PATH.resolve()}"

print(DB_PATH.resolve())
print(REVIEW_SUMMARIES_PATH.resolve())

C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\steam_games_reviews_25.sqlite
C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\review_summaries.csv


## Load game metadata and review summaries

In [2]:
def load_games_metadata(db_path: Path) -> pd.DataFrame: 
    conn = sqlite3.connect(db_path)
    query = """
    SELECT
        appid,
        name,
        short_description,
        about_the_game,
        genres_json,
        categories_json,
        tags_json
    FROM games
    WHERE name IS NOT NULL
      AND TRIM(name) != ''
    ORDER BY appid
    """
    try:
        return pd.read_sql_query(query, conn)
    finally:
        conn.close()


games_df = load_games_metadata(DB_PATH)
review_summaries_df = pd.read_csv(REVIEW_SUMMARIES_PATH)

print(f"Loaded {len(games_df):,} games.")
print(f"Loaded {len(review_summaries_df):,} review summaries.")
games_df.head()

Loaded 39,175 games.
Loaded 1,994 review summaries.


,appid,name,short_description,about_the_game,genres_json,categories_json,tags_json
0,10,Counter-Strike,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this w...,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this w...,"[""Action""]","[""Multi-player"", ""PvP"", ""Online PvP"", ""Shared/Split Screen PvP"", ""Color Alternatives"", ""Custom Volume Controls"", ""Ke...","{""Action"": 5504, ""FPS"": 4929, ""Multiplayer"": 3474, ""Shooter"": 3419, ""Classic"": 2849, ""Team-Based"": 1921, ""First-Pers..."
1,20,Team Fortress Classic,"One of the most popular online action games of all time, Team Fortress Classic features over nine character classes ...","One of the most popular online action games of all time, Team Fortress Classic features over nine character classes ...","[""Action""]","[""Multi-player"", ""PvP"", ""Online PvP"", ""Shared/Split Screen PvP"", ""Custom Volume Controls"", ""Keyboard Only Option"", ""...","{""Action"": 767, ""FPS"": 333, ""Multiplayer"": 280, ""Classic"": 254, ""Hero Shooter"": 226, ""Shooter"": 224, ""Team-Based"": 2..."
2,30,Day of Defeat,Enlist in an intense brand of Axis vs. Allied teamplay set in the WWII European Theatre of Operations. Players assum...,Enlist in an intense brand of Axis vs. Allied teamplay set in the WWII European Theatre of Operations. Players assum...,"[""Action""]","[""Multi-player"", ""Camera Comfort"", ""Color Alternatives"", ""Custom Volume Controls"", ""Stereo Sound"", ""Valve Anti-Cheat...","{""FPS"": 804, ""World War II"": 272, ""Multiplayer"": 216, ""Shooter"": 194, ""Action"": 165, ""War"": 159, ""Team-Based"": 140, ..."
3,40,Deathmatch Classic,Enjoy fast-paced multiplayer gaming with Deathmatch Classic (a.k.a. DMC). Valve's tribute to the work of id software...,Enjoy fast-paced multiplayer gaming with Deathmatch Classic (a.k.a. DMC). Valve's tribute to the work of id software...,"[""Action""]","[""Multi-player"", ""PvP"", ""Online PvP"", ""Shared/Split Screen PvP"", ""Color Alternatives"", ""Custom Volume Controls"", ""Ke...","{""Action"": 638, ""FPS"": 155, ""Classic"": 119, ""Multiplayer"": 109, ""Shooter"": 103, ""First-Person"": 83, ""Arena Shooter"":..."
4,50,Half-Life: Opposing Force,Return to the Black Mesa Research Facility as one of the military specialists assigned to eliminate Gordon Freeman. ...,Return to the Black Mesa Research Facility as one of the military specialists assigned to eliminate Gordon Freeman. ...,"[""Action""]","[""Single-player"", ""Multi-player"", ""Custom Volume Controls"", ""Adjustable Difficulty"", ""Playable without Timed Input"",...","{""FPS"": 934, ""Action"": 362, ""Classic"": 291, ""Sci-fi"": 285, ""Singleplayer"": 267, ""Shooter"": 251, ""First-Person"": 222,..."


## Parse JSON metadata fields

In [3]:
def parse_json_or_default(value: str | None, default: Any) -> Any: #json strings to python objects
    if not value:
        return default
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return default


def top_k_tags(tag_value: Any, k: int = 8) -> list[str]: #extract most important tags (by weight)
    if isinstance(tag_value, dict):
        return [tag for tag, _ in sorted(tag_value.items(), key=lambda x: x[1], reverse=True)[:k]]
    if isinstance(tag_value, list):
        return tag_value[:k]
    return []


games_df["genres"] = games_df["genres_json"].apply(lambda x: parse_json_or_default(x, []))
games_df["categories"] = games_df["categories_json"].apply(lambda x: parse_json_or_default(x, []))
games_df["tags"] = games_df["tags_json"].apply(lambda x: parse_json_or_default(x, {}))
games_df["top_tags"] = games_df["tags"].apply(lambda x: top_k_tags(x, k=8))

games_df[["name", "genres", "categories", "top_tags"]].head()

,name,genres,categories,top_tags
0,Counter-Strike,[Action],"[Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color Alternatives, Custom Volume Controls, Keyboard Only O...","[Action, FPS, Multiplayer, Shooter, Classic, Team-Based, First-Person, Competitive]"
1,Team Fortress Classic,[Action],"[Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Custom Volume Controls, Keyboard Only Option, Stereo Sound,...","[Action, FPS, Multiplayer, Classic, Hero Shooter, Shooter, Team-Based, Class-Based]"
2,Day of Defeat,[Action],"[Multi-player, Camera Comfort, Color Alternatives, Custom Volume Controls, Stereo Sound, Valve Anti-Cheat enabled, F...","[FPS, World War II, Multiplayer, Shooter, Action, War, Team-Based, Classic]"
3,Deathmatch Classic,[Action],"[Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color Alternatives, Custom Volume Controls, Keyboard Only O...","[Action, FPS, Classic, Multiplayer, Shooter, First-Person, Arena Shooter, Old School]"
4,Half-Life: Opposing Force,[Action],"[Single-player, Multi-player, Custom Volume Controls, Adjustable Difficulty, Playable without Timed Input, Save Anyt...","[FPS, Action, Classic, Sci-fi, Singleplayer, Shooter, First-Person, Aliens]"


## Merge review summaries into the games table

In [4]:
merged_df = games_df.merge(review_summaries_df, on="appid", how="left") #left join to keep all games, even those without reviews
merged_df["review_summary"] = merged_df["review_summary"].fillna("")
merged_df[["appid", "name", "review_summary"]].head()

,appid,name,review_summary
0,10,Counter-Strike,"[positive, helpful_votes=100] ---{ Graphics }---\n☐ You forget what reality is\n☐ Beautiful\n☐ Good\n☐ Decent\n☑ Bad..."
1,20,Team Fortress Classic,"[positive, helpful_votes=82] Since no one on TF community will read this review, I wish to the community, that inclu..."
2,30,Day of Defeat,"[positive, helpful_votes=88] I am 32 years old.\n\nMy ex-wife and I have a daughter together, and we adopted our son..."
3,40,Deathmatch Classic,"[positive, helpful_votes=84] If Deathmatch Classic has a million fans, I am one of them.\nIf Deathmatch Classic has ..."
4,50,Half-Life: Opposing Force,"[positive, helpful_votes=9] Half-Life: Opposing Force introduces players to Corporal Adrian Shephard, a member of th..."


## Build one retrieval text per game

In [5]:
def list_to_text(value: Any, max_items: int = 8) -> str: #convert python list into comma separated string, with max items to avoid too long texts
    if isinstance(value, list):
        return ", ".join(str(item) for item in value[:max_items])
    return ""


def clean_review_summary(text: str, max_chars: int = 1200) -> str: #trim
    return (text or "").strip()[:max_chars]


def build_retrieval_text(row: pd.Series) -> str:
    parts = [
        f"Name: {row['name']}".strip(),
        f"Genres: {list_to_text(row['genres'], max_items=6)}".strip(),
        f"Categories: {list_to_text(row['categories'], max_items=6)}".strip(),
        f"Tags: {', '.join(row['top_tags'])}".strip(),
        f"Short description: {(row['short_description'] or '').strip()}".strip(),
    ]

    about_text = (row["about_the_game"] or "").strip()
    if about_text:
        parts.append(f"About: {about_text[:500]}")

    review_text = clean_review_summary(row["review_summary"], max_chars=1200)
    if review_text:
        parts.append(f"Player feedback: {review_text}")

    return "\n".join(part for part in parts if part and not part.endswith(":"))


merged_df["retrieval_text"] = merged_df.apply(build_retrieval_text, axis=1) #every game has one final text representatio
merged_df[["appid", "name", "retrieval_text"]].head()

,appid,name,retrieval_text
0,10,Counter-Strike,"Name: Counter-Strike\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color Alte..."
1,20,Team Fortress Classic,"Name: Team Fortress Classic\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cus..."
2,30,Day of Defeat,"Name: Day of Defeat\nGenres: Action\nCategories: Multi-player, Camera Comfort, Color Alternatives, Custom Volume Con..."
3,40,Deathmatch Classic,"Name: Deathmatch Classic\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color ..."
4,50,Half-Life: Opposing Force,"Name: Half-Life: Opposing Force\nGenres: Action\nCategories: Single-player, Multi-player, Custom Volume Controls, Ad..."


## Inspect examples

In [6]:
pd.set_option("display.max_colwidth", None)
merged_df[["appid", "name", "retrieval_text"]].head(5)

,appid,name,retrieval_text
0,10,Counter-Strike,"Name: Counter-Strike\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color Alternatives, Custom Volume Controls\nTags: Action, FPS, Multiplayer, Shooter, Classic, Team-Based, First-Person, Competitive\nShort description: Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.\nAbout: Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.\nPlayer feedback: [positive, helpful_votes=100] ---{ Graphics }---\n☐ You forget what reality is\n☐ Beautiful\n☐ Good\n☐ Decent\n☑ Bad\n☐ Don‘t look too long at it\n☐ MS-DOS\n\n---{ Gameplay }---\n☑ Very good\n☐ Good\n☐ It's just gameplay\n☐ Mehh\n☐ Watch paint dry instead\n☐ Just don't\n\n---{ Audio }---\n☑ Eargasm\n☐ Very good\n☐ Good\n☐ Not too bad\n☐ Bad\n☐ I'm now deaf\n\n---{ Audience }---\n☑ Kids\n☑ Teens\n☑ Adults\n☑ Grandma\n\n---{ PC Requirements }---\n☑ Check if you can run paint\n☐ Potato\n☐ Decent\n☐ Fast\n☐ Rich boi\n☐ Ask NASA if they have a spare computer\n\n---{ Difficulty }---\n☐ Just press 'W'\n☐ Easy\n☑ Easy to learn / Hard to master\n☐ Significant brain usage\n☐ Difficult\n☐ Dark Souls\n\n---{ Grind }---\n☐ Nothing to grind\n☑ Only if u care about leaderboards/ranks\n☐ Isn't necessary to progress\n☐ Average grind level\n☐ Too much grind\n☐ You'll need a second life for grinding\n\n---{ Story }---\n☑ No Story\n☐ Some lore\n☐ Average\n☐ Good\n☐ Lovely\n☐ It'll replace your life\n\n---{ Game Time }---\n☐ Long enough for a cup of coffee\n☐ Short\n☐ Average\n☐ Long\n☑ To infinity and beyond\n\n---{ Price }---\n☐ It's free!\n☑ Worth the price\n☐ If it's on sale\n☐ If u have some spare money left\n☐ Not recommended\n☐ You could also just burn your money\n\n---{"
1,20,Team Fortress Classic,"Name: Team Fortress Classic\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Custom Volume Controls, Keyboard Only Option\nTags: Action, FPS, Multiplayer, Classic, Hero Shooter, Shooter, Team-Based, Class-Based\nShort description: One of the most popular online action games of all time, Team Fortress Classic features over nine character classes -- from Medic to Spy to Demolition Man -- enlisted in a unique style of online team warfare. Each character class possesses unique weapons, items, and abilities, as teams compete online in a variety of game play modes.\nAbout: One of the most popular online action games of all time, Team Fortress Classic features over nine character classes -- from Medic to Spy to Demolition Man -- enlisted in a unique style of online team warfare. Each character class possesses unique weapons, items, and abilities, as teams compete online in a variety of game play modes.\nPlayer feedback: [positive, helpful_votes=82] Since no one on TF community will read this review, I wish to the community, that includes the vets that has been playing on steam since 2003, Merry christmas and happy new year. <3.\n[positive, helpful_votes=48] Played for 10 minutes. Accidentally TK'd somebody. Then they spammed my IP address so I uninstalled just as fast as I downloaded this gem. 10/10\n[positive, helpful_votes=36] There' like 30 people in the world playing and they are all degenerates. Makes it fun.\n[negative, helpful_votes=9] It's 2025, and I'm stuck in a team shooter... with no team...\n\nTeam Fortress Classic is a class-based multiplayer shooter from 1999 that relies entirely on teamwork. In 2025, that dependency is its biggest weakness.\n\nMovement and

## Save to CSV

In [7]:
output_df = merged_df[
    [
        "appid",
        "name",
        "short_description",
        "genres_json",
        "categories_json",
        "tags_json",
        "review_summary",
        "retrieval_text",
    ]
].copy()

output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")

Saved to: C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\games_retrieval_texts.csv
